# FastLane small model: QLoRA on a free T4 (Colab or Kaggle)

Trains the LoRA adapter for Qwen2.5-3B-Instruct (or 1.5B) on the FastLane practice-call data, merges it, and converts it to GGUF for Ollama.

**Before running:** on your machine run `fastlane train data` then `fastlane train colab-bundle`, and upload `data/train/colab_bundle.zip` when the upload cell asks.

Runtime: GPU (T4). About 40–60 minutes per epoch for 3B at ~3.7k examples.

In [ ]:
!pip -q install -U "transformers>=4.45" "peft>=0.12" "trl>=0.20" datasets bitsandbytes accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import files  # On Kaggle, add the files as a dataset instead and skip this cell
uploaded = files.upload()  # colab_bundle.zip
!unzip -o -q colab_bundle.zip

In [ ]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # or 'Qwen/Qwen2.5-1.5B-Instruct'
ADAPTER = 'all_domains'              # or 'movies_buses' for variant A8
OUT = f'lora-{MODEL.split("/")[-1]}-{ADAPTER}'
!python train_lora.py --data data/{ADAPTER} --model {MODEL} --out {OUT} --epochs 2 --batch 2 --grad-accum 8 --merge

In [ ]:
!cat {OUT}/eval.json

## Convert to GGUF for Ollama

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py {OUT}/merged --outfile {OUT}.f16.gguf --outtype f16

In [ ]:
# Quantise to Q4_K_M (smaller and faster; same format as the Ollama base models)
!cd llama.cpp && cmake -B build -DGGML_CUDA=OFF > /dev/null && cmake --build build --target llama-quantize -j 4 > /dev/null
!llama.cpp/build/bin/llama-quantize {OUT}.f16.gguf {OUT}.Q4_K_M.gguf Q4_K_M

In [ ]:
!zip -q -r {OUT}-adapter.zip {OUT}/adapter {OUT}/eval.json
files.download(f'{OUT}.Q4_K_M.gguf')
files.download(f'{OUT}-adapter.zip')

## Back on your machine

```bash
fastlane train ollama-import path/to/lora-Qwen2.5-3B-Instruct-all_domains.Q4_K_M.gguf --name fastlane-qwen2.5-3b
```
This creates the Ollama model `fastlane-qwen2.5-3b`, which the `small_lora` role in `config.py` already points to. From then on FastLane uses the tuned model with its compact prompt automatically.